In [239]:
#from google.colab import drive
#drive.mount('/content/drive')

In [240]:
import sys
sys.path.append(r"C:\Users\micha\OneDrive\Documents\Python Scripts\open_sees\final_\functions")

In [241]:
#!sudo apt update
#!sudo apt install libglu1-mesa libglu1-mesa-dev
#!pip install -r requirements.txt

In [242]:
import openseespy.opensees as ops
import opsvis as opsv
import numpy as np
import ipywidgets as widgets
import os
import matplotlib.pyplot as plt
import math
import opstool as opst
import time as tt
import gmsh


In [243]:
import solver_function as SF
import materialParam as param
import importlib
importlib.reload(SF)
importlib.reload(param)
from solver_function import *
from materialParam import *

In [244]:
length_unit = "m"  # base unit
force_unit = "KN"  # base unit

UNIT = opst.pre.UnitSystem(length=length_unit, force=force_unit)

print("Length:", UNIT.mm, UNIT.mm2, UNIT.cm, UNIT.m,)
print("Force", UNIT.N, UNIT.kN, )
print("Stress", UNIT.MPa, UNIT.kPa, UNIT.Pa, )
print("Mass", UNIT.g, UNIT.kg, UNIT.ton)
print(UNIT)

Length: 0.001 1e-06 0.01 1
Force 0.001 1
Stress 1000.0 1.0 0.001
Mass 1e-06 0.001 1.0
<UnitSystem: length='m', force='kn', time='sec' (145837810245)>


In [245]:
import gmsh

# --- Initialize Gmsh ---
gmsh.initialize()
gmsh.model.add("erss_quadUP")
occ = gmsh.model.occ

# --- Geometry ---
rect1 = occ.addRectangle(0, 0, 0, 30, 6)
rect2 = occ.addRectangle(0, 6, 0, 30, 6)
rect3 = occ.addRectangle(0, 12, 0, 30, 4)
occ.removeAllDuplicates()
occ.synchronize()

surfs = [tag for dim, tag in gmsh.model.getEntities(2)]

# --- Transfinite mesh: surface 1 ---
gmsh.model.mesh.setTransfiniteCurve(4, 13)
gmsh.model.mesh.setTransfiniteCurve(2, 13)
gmsh.model.mesh.setTransfiniteCurve(3, 61)
gmsh.model.mesh.setTransfiniteCurve(1, 61)
gmsh.model.mesh.setTransfiniteSurface(1)
gmsh.model.mesh.setRecombine(2, 1)
occ.synchronize()

gmsh.model.mesh.setTransfiniteCurve(7, 16)
gmsh.model.mesh.setTransfiniteCurve(5, 16)
gmsh.model.mesh.setTransfiniteCurve(6, 61)
gmsh.model.mesh.setTransfiniteSurface(2)
gmsh.model.mesh.setRecombine(2, 2)
occ.synchronize()

gmsh.model.mesh.setTransfiniteCurve(10, 9)
gmsh.model.mesh.setTransfiniteCurve(8, 9)
gmsh.model.mesh.setTransfiniteCurve(9, 61)
gmsh.model.mesh.setTransfiniteSurface(3)
gmsh.model.mesh.setRecombine(2, 3)
occ.synchronize()

#-------------RIGHT SUPPORTS--------------#
entity_tags = [4,7,10]
physical_names = ["left_supp_a", "left_supp_b", "left_supp_c"]

base_tag = 201
for i, (tag, name) in enumerate(zip(entity_tags, physical_names)):
    physical_tag = base_tag + i
    gmsh.model.addPhysicalGroup(1, [tag], tag=physical_tag)
    gmsh.model.setPhysicalName(1, physical_tag, name)
    
#-------------LEFT SUPPORTS--------------#
entity_tags = [2,5,8]
physical_names = ["right_supp_a", "right_supp_b","right_supp_c"]

base_tag = 301
for i, (tag, name) in enumerate(zip(entity_tags, physical_names)):
    physical_tag = base_tag + i
    gmsh.model.addPhysicalGroup(1, [tag], tag=physical_tag)
    gmsh.model.setPhysicalName(1, physical_tag, name)

#-------------BOT SUPPORTS--------------#
entity_tags = [1]
physical_names = ["bot_supp_a"]

base_tag = 401
for i, (tag, name) in enumerate(zip(entity_tags, physical_names)):
    physical_tag = base_tag + i
    gmsh.model.addPhysicalGroup(1, [tag], tag=physical_tag)
    gmsh.model.setPhysicalName(1, physical_tag, name)

#-------------SOIL NAMES--------------#
base_tag = 701

# Create a list of entity tags from 1 to 20
entity_tags = list(range(1, 6))

# Create a list of 20 unique physical names
physical_names = [f"soil_{chr(97 + i)}" for i in range(6)]

soil_tags = []
soil_names = []

# Iterate through the lists using the zip() function
for i, (tag, name) in enumerate(zip(entity_tags, physical_names)):
    physical_tag = base_tag + i
    gmsh.model.addPhysicalGroup(2, [tag], tag=physical_tag)
    gmsh.model.setPhysicalName(2, physical_tag, name)

    # Store the generated tag and name in our new lists
    soil_tags.append(physical_tag)
    soil_names.append(name)

# --- Finalize mesh ---
gmsh.model.occ.synchronize()
gmsh.model.mesh.generate(2)
gmsh.write("erss_quadUP.msh")
# gmsh.fltk.run()
# gmsh.finalize()

In [246]:
GMSH2OPS = opst.pre.Gmsh2OPS(ndm=2, ndf=2)
GMSH2OPS.read_gmsh_file("erss_quadUP.msh")
gmsh.finalize()

Info:: 12 Physical Names.
Info:: 2196 Nodes; MaxNodeTag 2196; MinNodeTag 1.
Info:: 2230 Elements; MaxEleTag 2230; MinEleTag 1.
Info:: Geometry Information >>>
21 Entities: 8 Point; 10 Curves; 3 Surfaces; 0 Volumes.

Info:: Physical Groups Information >>>
10 Physical Groups.
Physical Group names: ['bot_supp_a', 'right_supp_a', 'left_supp_a', 'right_supp_b', 'left_supp_b', 'right_supp_c', 'left_supp_c', 'soil_a', 'soil_b', 'soil_c']

Info:: Mesh Information >>>
2196 Nodes; MaxNodeTag 2196; MinNodeTag 1.
2230 Elements; MaxEleTag 2230; MinEleTag 1.



In [247]:
GMSH2OPS.get_physical_groups()

{'bot_supp_a': [(1, 1)],
 'right_supp_a': [(1, 2)],
 'left_supp_a': [(1, 4)],
 'right_supp_b': [(1, 5)],
 'left_supp_b': [(1, 7)],
 'right_supp_c': [(1, 8)],
 'left_supp_c': [(1, 10)],
 'soil_a': [(2, 1)],
 'soil_b': [(2, 2)],
 'soil_c': [(2, 3)]}

In [248]:
ops.wipe()
ops.model("basic", "-ndm", 2, "-ndf", 3)

In [249]:
medium = param.create_material(ops, param.medium_sand_params, matTag=10, dof=2)
medium_dense = param.create_material(ops, param.medium_dense_sand_params, matTag=11, dof=2)
dense = param.create_material(ops, param.dense_sand_params, matTag=12, dof=2)

ops.nDMaterial("InitialStateAnalysisWrapper", 1, 10, 2)
ops.nDMaterial("InitialStateAnalysisWrapper", 2, 11, 2)
ops.nDMaterial("InitialStateAnalysisWrapper", 3, 12, 2)


Will use 10 yield surfaces.
Will use 10 yield surfaces.
Will use 10 yield surfaces.


In [250]:
GMSH2OPS.create_node_cmds()
vperm = 5.0e-9  # vertical permeability (m/s) for soft
hperm = 5.0e-9  # horizontal permeability (m/s) for soft
accGravity = 9.81  # acceleration of gravity
bulkWater  = 2.2e6 #kPa
medium_soil_density = 19 #kN/m^3
E_medium_soil = 10*UNIT.MPA #mpa
voidratio_medium = 0.7
medium_alpha = SF.compute_alpha(h = 0.5 , density = medium_soil_density, youngs_modulus = E_medium_soil)
# Actual values used in computation
vperm = vperm / accGravity / fmass
hperm = hperm / accGravity / fmass

soil_groups = [
    "soil_a",
    "soil_b",
    "soil_c",
   
]
#           MatTag, thick,
element_args = [ 1, 1.0,  bulkWater, 1.0, hperm, hperm, voidratio_medium, medium_alpha, 0.0, -9.81] # press, rho_d, unitWeightX, unitWeightY

# Create elements for each soil group
for group_name in soil_groups:
    ele_tags = GMSH2OPS.create_element_cmds(
        ops_ele_type="SSPquadUP",
        ops_ele_args=element_args,
        physical_group_names=[group_name]
    )

In [251]:
ele_rem = list(range(1791,1799))
print(ele_rem)
for mm in ele_rem:
    ops.remove('ele', mm)

[1791, 1792, 1793, 1794, 1795, 1796, 1797, 1798]


In [252]:
opst.vis.plotly.plot_model(show_node_numbering=False, show_ele_numbering=False, show_nodal_loads = False)

In [212]:
opst.vis.plotly.plot_model(show_node_numbering=False, show_ele_numbering=True, show_nodal_loads = False)